# 🚀 LLaMA 7B Fine-tuning for Sharekhan Broking Domain

This notebook fine-tunes LLaMA 2 7B using Unsloth on Sharekhan FAQs for customer support.

**Before running:**
1. Go to `Runtime` → `Change runtime type`
2. Select `T4 GPU` as Hardware accelerator
3. Click `Save`
4. Upload `sharekhan_faqs_training.json` to Colab

---

## Step 1: Install Dependencies
This will take 2-3 minutes

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

# Disable wandb
import os
os.environ["WANDB_DISABLED"] = "true"

print("✅ Installation complete!")

## Step 2: Check GPU

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 3: Load Base Model
Loading LLaMA 2 7B with 4-bit quantization

In [ ]:
from unsloth import FastLanguageModel

# Configuration
max_seq_length = 2048
dtype = None  # Auto-detect
load_in_4bit = True  # Use 4-bit quantization

# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-2-7b-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("✅ Model loaded successfully!")

## Step 4: Add LoRA Adapters
Adding trainable adapters to the model

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("✅ LoRA adapters added!")

## Step 5: Upload and Load Sharekhan Training Data
Upload the sharekhan_faqs_training.json file first

In [ ]:
# Upload the training data file
from google.colab import files
print("📤 Please upload sharekhan_faqs_training.json file...")
uploaded = files.upload()
print("✅ File uploaded successfully!")

In [ ]:
import json

# Alpaca prompt template
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

# Load Sharekhan training data from uploaded file
with open('sharekhan_faqs_training.json', 'r', encoding='utf-8') as f:
    training_data = json.load(f)

print(f"✅ Loaded {len(training_data)} Sharekhan FAQ training examples")

In [ ]:
from datasets import Dataset

# Format data with Alpaca template
def format_example(example):
    text = alpaca_prompt.format(
        instruction=example["instruction"],
        input=example["input"],
        output=example["output"],
    )
    return {"text": text}

formatted_data = [format_example(ex) for ex in training_data]
dataset = Dataset.from_list(formatted_data)

print(f"✅ Dataset prepared with {len(dataset)} examples")
print(f"\nSample formatted prompt:\n{dataset[0]['text'][:500]}...")

## Step 6: Train the Model
This will take ~10-15 minutes on T4 GPU with 60+ examples

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        output_dir="./outputs",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_steps=5,
        logging_steps=1,
        save_steps=50,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        report_to="none",
    ),
)

print("Starting training...")
trainer_stats = trainer.train()

print(f"\n✅ Training completed!")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.2f}s")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")

## Step 7: Test the Model
Let's test our fine-tuned Sharekhan FAQ model!

In [ ]:
# Enable inference mode
FastLanguageModel.for_inference(model)

# Test prompt - Sharekhan specific question
test_prompt = alpaca_prompt.format(
    instruction="How can I check my KRA Status?",
    input="",
    output="",
)

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("=" * 50)
print("MODEL RESPONSE:")
print("=" * 50)
print(response.split("### Response:")[-1].strip())

In [ ]:
# Test another Sharekhan-specific prompt
test_prompt2 = alpaca_prompt.format(
    instruction="What is Margin Trading Facility (MTF)?",
    input="",
    output="",
)

inputs = tokenizer(test_prompt2, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.7, do_sample=True)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("=" * 50)
print("MODEL RESPONSE:")
print("=" * 50)
print(response.split("### Response:")[-1].strip())

In [ ]:
# Test API-related question
test_prompt3 = alpaca_prompt.format(
    instruction="Can I use Sharekhan API in multiple programming languages?",
    input="",
    output="",
)

inputs = tokenizer(test_prompt3, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.7, do_sample=True)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("=" * 50)
print("MODEL RESPONSE:")
print("=" * 50)
print(response.split("### Response:")[-1].strip())

## Step 8: Save Model & Download to Your Computer
Save model locally and download it

In [ ]:
import os

# Create local output directory
save_path = "./sharekhan-llama-7b"
os.makedirs(save_path, exist_ok=True)

print(f"Saving model to: {save_path}")

In [ ]:
# Save LoRA adapters (small, ~50MB)
lora_path = f"{save_path}/lora_adapters"
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)
print(f"✅ LoRA adapters saved to: {lora_path}")

In [ ]:
# Save to GGUF format (for llama.cpp/Ollama deployment)
# This creates a quantized model that runs on CPU too!

print("Converting to GGUF format (this may take a few minutes)...")
model.save_pretrained_gguf(
    save_path,
    tokenizer,
    quantization_method="q4_k_m"  # Good balance of quality and size (~4GB)
)
print(f"✅ GGUF model saved!")

In [ ]:
# List saved files
import os
print("\n📁 Saved files:")
for root, dirs, files in os.walk(save_path):
    for file in files:
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath) / (1024*1024)  # MB
        print(f"  {file}: {size:.1f} MB")

In [ ]:
# Zip the LoRA adapters for easy download (FIXED - using shutil instead of shell command)
import shutil
shutil.make_archive('lora_adapters', 'zip', save_path, 'lora_adapters')
print("✅ Created lora_adapters.zip")

In [ ]:
# Download files to your local computer
from google.colab import files

print("📥 Downloading LoRA adapters (small, ~50MB)...")
files.download('lora_adapters.zip')

In [ ]:
# Download GGUF file (larger, ~4GB)
# Find the GGUF file
import glob
gguf_files = glob.glob(f"{save_path}/*.gguf")
if gguf_files:
    gguf_file = gguf_files[0]
    print(f"📥 Downloading GGUF file: {gguf_file}")
    print("⚠️ This is a large file (~4GB), download may take a while...")
    files.download(gguf_file)
else:
    print("No GGUF file found")

## ✅ Done!

Your fine-tuned Sharekhan FAQ model is ready for download!

**Downloaded files:**
- `lora_adapters.zip` - LoRA adapters (~50MB) - use with transformers
- `*.gguf` - Quantized model (~4GB) - use with Ollama/llama.cpp

**Next Steps for GGUF (Ollama):**
1. Install Ollama: https://ollama.ai
2. Create a Modelfile:
```
FROM ./sharekhan-llama-7b-q4_k_m.gguf
TEMPLATE """### Instruction:\n{{.Prompt}}\n\n### Response:\n"""
```
3. Run: `ollama create sharekhan-llama -f Modelfile`
4. Chat: `ollama run sharekhan-llama`